In [1]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df=pd.read_csv("lahore_flats_final.csv")

In [3]:
df.columns

Index(['Property ID', 'Society', 'Society Link', 'Name', 'Page Title', 'Price',
       'Area', 'Bedrooms', 'Baths', 'Floor Number', 'Total Floors',
       'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Built in year', 'Parking Spaces', 'Lobby in Building',
       'Double Glazed Windows', 'Central Air Conditioning', 'Central Heating',
       'Flooring', 'Electricity Backup', 'Waste Disposal', 'Floor',
       'Floors in Building', 'Elevators', 'Service Elevators in Building',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Community Lawn or Garden', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre', 'Nearby Schools',
       'Nearby Hospitals', 'Nearby Shopping Malls', 'Nearby Restaurants',
       'Distance From Airport (kms)', 'Nearby Public Transport Service',
       'Other N

In [28]:
# 2. Description-based society df
society_desc_df = df.groupby('Society').agg({
    'Description': ' '.join
}).reset_index()

In [5]:
df['Features'].value_counts()

Features
Furnished | Broadband Internet Access | Satellite or Cable TV Ready | Business Center or Media Room in Building | Conference Room in Building | Intercom | ATM Machines | Community Lawn or Garden | Community Swimming Pool | Community Gym | First Aid or Medical Centre | Day Care Centre | Kids Play Area | Barbeque Area | Mosque | Community Centre | Sauna | Jacuzzi | Nearby Schools | Nearby Hospitals | Nearby Shopping Malls | Nearby Restaurants | Nearby Public Transport Service | Other Nearby Places | Maintenance Staff | Security Staff | Laundry or Dry Cleaning Facility | Communal/Shared Kitchen | Facilities for Disabled                                                                                                                                                                                                                                                                                                                                                                               

In [8]:
import re

def clean_features(text):

    if pd.isna(text):
        return ""

    text = str(text)

    remove_patterns = [
        r'Built in year\s*:\s*\d+',
        r'Parking Spaces\s*:\s*\d*',
        r'Floor\s*:\s*\d*',
        r'Floors in Building\s*:\s*\d*',
        r'Elevators\s*:\s*\d*',
        r'Distance From Airport \(kms\)',
        r'Other Facilities',
        r'Other Nearby Places',
        r'Other Main Features',
        r'Other Community Facilities',
        r'Other Healthcare and Recreation Facilities',
        r'Other Business and Communication Facilities',
        r'Waste Disposal',
        r'Service Elevators in Building',
        r'Communal/Shared Kitchen',
        r'\bFloor\b',
        r'\bFloors in Building\b',
        r'\bParking Spaces\b'
    ]

    # remove unwanted text
    for pattern in remove_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # remove multiple separators
    text = re.sub(r'(\|\s*)+', '| ', text)

    # remove repeated spaces
    text = re.sub(r'\s+', ' ', text)

    # clean start/end pipes
    text = text.strip(' |')

    return text


# apply
df['clean_features'] = df['Features'].apply(clean_features)

In [9]:
df['clean_features'].value_counts()

clean_features
Lobby in Building | Double Glazed Windows | Central Air Conditioning | Central Heating | Flooring | Electricity Backup | Furnished | Broadband Internet Access | Satellite or Cable TV Ready | Business Center or Media Room in Building | Conference Room in Building | Intercom | ATM Machines | Community Lawn or Garden | Community Swimming Pool | Community Gym | First Aid or Medical Centre | Day Care Centre | Kids Play Area | Barbeque Area | Mosque | Community Centre | Sauna | Jacuzzi | Nearby Schools | Nearby Hospitals | Nearby Shopping Malls | Nearby Restaurants | Nearby Public Transport Service | Maintenance Staff | Security Staff | Laundry or Dry Cleaning Facility | Facilities for Disabled | Pets Allowed    203
Furnished | Broadband Internet Access | Satellite or Cable TV Ready | Business Center or Media Room in Building | Conference Room in Building | Intercom | ATM Machines | Community Lawn or Garden | Community Swimming Pool | Community Gym | First Aid or Medical Centr

In [30]:
society_feature_df = df.groupby('Society').agg({
    'clean_features': ' '.join
}).reset_index()


In [34]:
# convert clean_features column into simple python lists

society_feature_df['feature_list'] = society_feature_df['clean_features'].apply(
    lambda x: [i.strip() for i in str(x).split('|') if i.strip()]
)

In [35]:
society_feature_df['feature_list'].value_counts()

feature_list
[Lobby in Building, Double Glazed Windows, Central Air Conditioning, Central Heating, Flooring, Electricity Backup, Furnished, Broadband Internet Access, Satellite or Cable TV Ready, Business Center or Media Room in Building, Conference Room in Building, Intercom, ATM Machines, Community Lawn or Garden, Community Swimming Pool, Community Gym, First Aid or Medical Centre, Day Care Centre, Kids Play Area, Barbeque Area, Mosque, Community Centre, Sauna, Jacuzzi, Nearby Schools, Nearby Hospitals, Nearby Shopping Malls, Nearby Restaurants, Nearby Public Transport Service, Maintenance Staff, Security Staff, Laundry or Dry Cleaning Facility, Facilities for Disabled, Pets Not Allowed]                                                                                                                                                                                                                                                                                                              

In [36]:
society_feature_df['freatureStr']=society_feature_df['feature_list'].apply(''.join)

In [37]:
df['freatureStr'].iloc[0]

'Lobby in BuildingDouble Glazed WindowsCentral Air ConditioningCentral HeatingFlooringElectricity BackupBroadband Internet AccessSatellite or Cable TV ReadyCommunity Lawn or GardenCommunity GymFirst Aid or Medical CentreDay Care CentreKids Play AreaBarbeque AreaMosqueCommunity CentreNearby SchoolsNearby HospitalsNearby Shopping MallsNearby RestaurantsNearby Public Transport ServiceMaintenance StaffSecurity Staff'

## vectorization

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity




In [39]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2)
)

In [40]:
tfidf_matrix = tfidf_vectorizer.fit_transform(society_feature_df['freatureStr'])

In [41]:
tfidf_matrix.toarray

<bound method _cs_matrix.toarray of <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9596 stored elements and shape (71, 994)>>

In [42]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [43]:
cosine_sim1.shape

(71, 71)

In [48]:
def recommend_societies(society_name, cosine_sim=cosine_sim1):

    # get index
    idx = society_feature_df[
        society_feature_df['Society'] == society_name
    ].index[0]

    # similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # sort
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # top 5 similar societies
    sim_scores = sim_scores[1:6]

    # indices
    society_indices = [i[0] for i in sim_scores]

    # dataframe
    recommendations_df = pd.DataFrame({
        'Society': society_feature_df['Society'].iloc[society_indices],
        'SimilarityScore': [i[1] for i in sim_scores]
    })

    return recommendations_df

In [57]:
society_feature_df['Society'].sample(10)

70                  Wapda Town
0       Abdul Sattar Edhi Road
69       Waheed Brother Colony
17                 Garden Town
22                  Izmir Town
29                  LDA Avenue
24                  Johar Town
61                  Shama Road
45    Pak Arab Housing Society
18                     Gulberg
Name: Society, dtype: object

In [58]:
recommend_societies("Gulberg")

,Society,SimilarityScore
12,DHA Defence,0.995682
35,Main Canal Bank Road,0.994034
13,Defence Road,0.986070
53,Raiwind Road,0.984579
24,Johar Town,0.982341
